## Step 1 – Load data

In [1]:
import warnings; warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, torch, torch.nn.functional as F
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score

torch.manual_seed(42); np.random.seed(42)

df = pd.read_csv("../data/framingham.csv")
df.loc[(df.currentSmoker == 0) & (df.cigsPerDay.isna()), "cigsPerDay"] = 0
df = df.drop(columns=["education"])
X = df.drop(columns=["TenYearCHD"]); y = df["TenYearCHD"]

X_train, X_tmp, y_train, y_tmp = train_test_split(X, y, test_size=0.4, stratify=y, random_state=42)
X_cal, X_test, y_cal, y_test = train_test_split(X_tmp, y_tmp, test_size=0.5, stratify=y_tmp, random_state=42)

imp = SimpleImputer(strategy="median").fit(X_train)
sc = StandardScaler().fit(imp.transform(X_train))

X_all = pd.concat([X_train, X_cal, X_test]); y_all = pd.concat([y_train, y_cal, y_test])
Z = sc.transform(imp.transform(X_all)).astype("float32")

n, n_tr, n_cal = len(Z), len(X_train), len(X_cal)
train_mask = torch.zeros(n, dtype=torch.bool); train_mask[:n_tr] = True
cal_mask   = torch.zeros(n, dtype=torch.bool); cal_mask[n_tr:n_tr+n_cal] = True
test_mask  = torch.zeros(n, dtype=torch.bool); test_mask[n_tr+n_cal:] = True
print(n, train_mask.sum().item(), cal_mask.sum().item(), test_mask.sum().item())

4238 2542 848 848


## Step 2 – Build the patient-similarity graph

In [2]:
from sklearn.neighbors import kneighbors_graph
from torch_geometric.data import Data
from torch_geometric.utils import to_undirected

K = 10   # each patient links to their 10 most similar patients
A = kneighbors_graph(Z, n_neighbors=K, include_self=False).tocoo()
edge_index = to_undirected(torch.tensor(np.vstack([A.row, A.col]), dtype=torch.long))

graph = Data(x=torch.tensor(Z), edge_index=edge_index,
             y=torch.tensor(y_all.values, dtype=torch.float))
print(graph)

Data(x=[4238, 14], edge_index=[2, 60604], y=[4238])


## Step 3 – Graph Attention Network (GAT)

In [3]:
from torch_geometric.nn import GATConv

class GAT(torch.nn.Module):
    def __init__(self, in_dim, hidden=32):
        super().__init__()
        self.conv1 = GATConv(in_dim, hidden, heads=4, dropout=0.3)
        self.conv2 = GATConv(hidden * 4, 1, heads=1, dropout=0.3)
    def forward(self, x, edge_index, return_attention=False):
        x = F.dropout(x, 0.3, self.training)
        x = F.elu(self.conv1(x, edge_index))
        x = F.dropout(x, 0.3, self.training)
        if return_attention:
            out, att = self.conv2(x, edge_index, return_attention_weights=True)
            return out.squeeze(-1), att
        return self.conv2(x, edge_index).squeeze(-1)

model = GAT(Z.shape[1])
opt = torch.optim.Adam(model.parameters(), lr=0.005, weight_decay=5e-4)
pos_weight = torch.tensor((y_train == 0).sum() / (y_train == 1).sum(), dtype=torch.float)

## Step 4 – Train and evaluate

In [4]:
best_auc, best_state = 0, None
for epoch in range(1, 301):
    model.train(); opt.zero_grad()
    out = model(graph.x, graph.edge_index)
    loss = F.binary_cross_entropy_with_logits(out[train_mask], graph.y[train_mask], pos_weight=pos_weight)
    loss.backward(); opt.step()

    model.eval()
    with torch.no_grad():
        p = torch.sigmoid(model(graph.x, graph.edge_index))
    cal_auc = roc_auc_score(graph.y[cal_mask], p[cal_mask])
    if cal_auc > best_auc:
        best_auc, best_state = cal_auc, {k: v.clone() for k, v in model.state_dict().items()}

model.load_state_dict(best_state); model.eval()
with torch.no_grad():
    p_gnn = torch.sigmoid(model(graph.x, graph.edge_index))
print("GNN test AUC:", round(roc_auc_score(graph.y[test_mask], p_gnn[test_mask]), 3))
print("Logistic regression baseline: 0.696")

GNN test AUC: 0.688
Logistic regression baseline: 0.696


## Step 5 – Explain a prediction using similar patients

In [5]:
# Get attention weights from the last GAT layer
with torch.no_grad():
    logits, (att_edges, att_w) = model(graph.x, graph.edge_index, return_attention=True)
att_w = att_w.squeeze(-1)
src, dst = att_edges

X_all_raw = X_all.reset_index(drop=True)
y_all_np = y_all.values

# Pick the highest-risk patient in the TEST set
test_idx = torch.where(test_mask)[0]
i = int(test_idx[torch.argmax(p_gnn[test_mask])])

print(f"Patient #{i}: predicted risk {p_gnn[i]*100:.0f}%  |  actually developed CHD: "
      f"{'Yes' if y_all_np[i] == 1 else 'No'}")
print(X_all_raw.loc[i, ["male", "age", "cigsPerDay", "sysBP", "totChol", "glucose", "diabetes"]].to_dict())

# Neighbours that point to this patient, keeping only TRAINING patients (outcomes known)
m = (dst == i) & (src != i)
neighbours = pd.DataFrame({"neighbour": src[m].numpy(), "attention": att_w[m].numpy()})
neighbours = neighbours[neighbours["neighbour"] < n_tr].sort_values("attention", ascending=False)
neighbours = neighbours.join(X_all_raw[["male", "age", "cigsPerDay", "sysBP", "totChol", "glucose"]],
                             on="neighbour")
neighbours["had_CHD"] = y_all_np[neighbours["neighbour"]]

print(f"\n{neighbours['had_CHD'].sum()} of {len(neighbours)} similar past patients developed heart disease")
neighbours.round(3)

Patient #3580: predicted risk 93%  |  actually developed CHD: Yes
{'male': 0.0, 'age': 67.0, 'cigsPerDay': 0.0, 'sysBP': 204.0, 'totChol': 303.0, 'glucose': 394.0, 'diabetes': 1.0}

4 of 5 similar past patients developed heart disease


,neighbour,attention,male,age,cigsPerDay,sysBP,totChol,glucose,had_CHD
0,868,0.097,1,62,0.0,102.5,346.0,394.0,1
2,1052,0.093,0,47,1.0,197.0,160.0,320.0,1
4,2469,0.091,0,61,0.0,200.0,265.0,256.0,1
1,1044,0.088,1,65,0.0,235.0,240.0,297.0,1
3,1292,0.081,1,64,0.0,176.0,195.0,370.0,0


## Summary

| Model | Test AUC |
|---|---|
| Logistic Regression | 0.696 |
| XGBoost | 0.679 |
| GNN (GAT, K=10) | 0.688 |

- GNN accuracy is **on par** with the baseline: graph links come from the same features, so no new information is added (K = 5/10/20 and GCN vs GAT all scored 0.688–0.695)
- GNN adds **case-based explainability**: attention weights identify the most influential similar past patients
- Example: highest-risk test patient (female, 67, glucose 394, SBP 204) – 4 of her 5 most similar past patients developed CHD, consistent with the SHAP explanation
- Next: federated GNN across the 4 UCI hospitals